# 0.26 — One preprocessed corpus for the whole pipeline

Until now every notebook assembled its own corpus from per-year `terms_{year}.parquet` files with
**mixed provenance** (legacy CaptureTime dating vs the fixed ADD-event dating of 0.25). This notebook
fixes the file layout once:

| layer | file | role |
|---|---|---|
| raw (immutable) | `data/raw/raw_news_{year}.csv.xz` | Bloomberg feed capture, never modified |
| per-year cache (disposable) | `output/tmp/terms_{year}.parquet` | built by `scripts/preprocess_news.py` — publications only (`ADD_1STPASS`/`ADD_STORY`), date = earliest ADD (rule `add-event-v1`, proof in 0.25); safe to delete, never read by analyses |
| **the corpus** | **`output/news_corpus.parquet`** | **single file, all years, cross-year deduped (first publication wins) — the only file downstream notebooks read** |
| build record | `output/news_corpus_manifest.json` | which years are certified under which rule; corpus staleness |

The first cell **runs the preprocessing if (and only if) needed** — any year missing from the
manifest is rebuilt from raw, and the corpus is re-merged when stale. Re-running the notebook when
everything is built takes seconds.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
import polars as pl
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(_ROOT / "scripts"))
import preprocess_news as pp

# THE gate: builds any year not certified under the current rule, re-merges if stale (idempotent)
CORPUS = pp.ensure()
manifest = json.loads(pp.MANIFEST.read_text())
print("\nrule:", manifest["rule"], "· corpus:", manifest["corpus"])

2010: certified add-event-v1 build exists — skipping
2011: certified add-event-v1 build exists — skipping
2012: certified add-event-v1 build exists — skipping
2013: certified add-event-v1 build exists — skipping
2014: certified add-event-v1 build exists — skipping
2015: certified add-event-v1 build exists — skipping
2016: certified add-event-v1 build exists — skipping
2017: certified add-event-v1 build exists — skipping
2018: certified add-event-v1 build exists — skipping
2019: certified add-event-v1 build exists — skipping
2020: certified add-event-v1 build exists — skipping
2021: certified add-event-v1 build exists — skipping
2022: certified add-event-v1 build exists — skipping
2023: certified add-event-v1 build exists — skipping
2024: certified add-event-v1 build exists — skipping
2025: certified add-event-v1 build exists — skipping
corpus up to date: news_corpus.parquet

rule: add-event-v1 · corpus: {'rows': 22626656, 'years': [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 

## The corpus in one look

Everything below reads **only** `news_corpus.parquet` — lazily, so no notebook ever needs to load
all years into memory.

In [2]:
lf = pl.scan_parquet(CORPUS)
per_year = (lf.group_by(pl.col("date").dt.year().alias("year")).len()
              .sort("year").collect())
print(per_year.to_pandas().to_string(index=False))
lo, hi, n = lf.select([pl.col("date").min().alias("lo"), pl.col("date").max().alias("hi"), pl.len()]).collect().row(0)
print(f"\n{n:,} headlines · {lo.date()} -> {hi.date()}")

 year     len
 2010 2179885
 2011 2083561
 2012 2082291
 2013 2022224
 2014 1973366
 2015 1873857
 2016 1552306
 2017 1193624
 2018 1057488
 2019  965687
 2020  985263
 2021 1013147
 2022  950419
 2023  899318
 2024  905755
 2025  888465

22,626,656 headlines · 2010-01-01 -> 2025-12-31


In [ ]:
# acceptance check 1 — the 0.25 case study: June-2013 "hack*" must sit at background level
h13 = (lf.filter((pl.col("date").dt.year() == 2013) &
                 pl.col("Headline").str.contains(r"(?i)\bhack(ed|ing|ers?)\b"))
         .group_by(pl.col("date").dt.month().alias("month")).len().sort("month").collect())
print("monthly hack* headlines, 2013 (was 1,365 in June under CaptureTime dating):")
print(h13.to_pandas().to_string(index=False))
assert h13.filter(pl.col("month") == 6)["len"][0] < 200, "June 2013 replay spike is back?!"
print("\nOK — replay spike gone")

In [4]:
# acceptance check 2 — top single-day volumes should be real news days, not bulk re-tag days
top = (lf.group_by(pl.col("date").dt.date().alias("day")).len()
         .sort("len", descending=True).head(5).collect())
print("biggest days in the corpus (should be genuine heavy-news days):")
print(top.to_pandas().to_string(index=False))

biggest days in the corpus (should be genuine heavy-news days):
       day   len
2010-10-28 14667
2012-04-26 14623
2011-10-27 14606
2011-04-28 14543
2010-04-28 14359


## How downstream notebooks should load it

One file, sliced lazily by date — replaces every `pd.concat([...terms_{y}.parquet...])` pattern:

```python
import polars as pl
corpus = (pl.scan_parquet(OUTPUT_DIR / "news_corpus.parquet")
            .filter(pl.col("date").is_between(pl.datetime(2015, 8, 1), pl.datetime(2018, 1, 31)))
            .collect().to_pandas())                      # [Headline, date, terms]

from theme_detect import detect_theme
df = detect_theme(corpus, "2018-01-17", theme_hint=r"bitcoin|\bcrypto|blockchain")
```

**Caveats after the rebuild:** all corpus dates changed vs the legacy files, so 0.24-era golden
outputs (e.g., `quantum_weekly_CHPX_etf.parquet`, previously reproduced exactly by the detect demo,
now 0.28) no longer match by design and need regenerating. The corpus also now starts in **2010**,
giving detection windows three extra baseline years.